# Configure dataset display and Spark conversion

`DatasetConfig` controls HypEx presentation and some limits on operations
that use driver memory. This notebook starts on pandas and uses local Spark
for the second section. Java 8, 11 or 17 is required for that section.

- [Change the dataset preview](#display)
- [Try the Spark-to-pandas conversion guard](#conversion)

<a id="display"></a>

## 1. Change the dataset preview

Display settings affect the representation of a `Dataset`. The underlying
rows and columns stay available. Configuration lives on the class and is
shared by the Python process. Use `patch.multiple` from the standard library
to apply temporary settings and restore the previous values when the block ends.

In [1]:
import os

# Local tutorial settings. Set these before importing PySpark or HypEx.
os.environ.setdefault("SPARK_LOCAL_IP", "127.0.0.1")
os.environ.setdefault("PYARROW_IGNORE_TIMEZONE", "1")

from unittest.mock import patch

from hypex.config import DatasetConfig
from hypex.dataset import Dataset, InfoRole, TargetRole
from hypex.utils import BackendsEnum

data = Dataset(
    roles={
        "user_id": InfoRole(int),
        "segment": InfoRole(str),
        "revenue": TargetRole(float),
        "visits": InfoRole(int),
    },
    data={"data": {
        "user_id": list(range(12)),
        "segment": ["new", "returning"] * 6,
        "revenue": [10.0 + i for i in range(12)],
        "visits": [i % 4 for i in range(12)],
    }},
)

with patch.multiple(DatasetConfig, DISPLAY_ROWS=2, DISPLAY_COLS=3):
    display(data)

print(f"The dataset still has {len(data)} rows and {len(data.columns)} columns.")

,user_id,segment,revenue,visits
0,0,new,10.0,0
1,1,returning,11.0,1
...,...,...,...,...
10,10,new,20.0,2
11,11,returning,21.0,3


The dataset still has 12 rows and 4 columns.


<a id="conversion"></a>

## 2. Try the Spark-to-pandas conversion guard

Move the small dataset to Spark and set an intentionally low threshold.
`SPARK_PANDAS_CONVERSION_LIMIT` bounds the number of rows HypEx can collect
through this conversion. A successful conversion at the threshold shows
that the limit is inclusive. The row count alone cannot determine driver
memory use, which also depends on column widths and types.

For a larger input, keep the analysis distributed or aggregate it on Spark
before collecting a small result.

In [2]:
from pathlib import Path
from tempfile import TemporaryDirectory


from pyspark.sql import SparkSession

# Configure the local JVM before Spark starts, while keeping error messages visible.
spark_files = TemporaryDirectory(prefix="hypex-spark-")
log_config = Path(spark_files.name) / "log4j2.properties"
log_config.write_text(
    "rootLogger.level = error\n"
    "rootLogger.appenderRef.console.ref = console\n"
    "appender.console.type = Console\n"
    "appender.console.name = console\n"
    "appender.console.target = SYSTEM_ERR\n"
    "appender.console.layout.type = PatternLayout\n"
    "appender.console.layout.pattern = %p %c: %m%n\n",
    encoding="utf-8",
)
spark = (
    SparkSession.builder.master("local[2]")
    .appName("HypEx DatasetConfig")
    .config("spark.ui.showConsoleProgress", "false")
    .config("spark.driver.extraJavaOptions", f"-Dlog4j.configurationFile={log_config.as_uri()}")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")

In [3]:
try:
    distributed = data.to_backend(BackendsEnum.spark, session=spark)
    display(distributed.select(["segment", "visits"]).value_counts())

    with patch.multiple(DatasetConfig, SPARK_PANDAS_CONVERSION_LIMIT=4):
        try:
            distributed.to_backend(BackendsEnum.pandas)
        except ValueError as error:
            if "exceed limit 4" not in str(error):
                raise
            print(error)
        else:
            # The guard must reject a collection larger than the configured limit.
            raise AssertionError("The full conversion should exceed the limit.")

    with patch.multiple(DatasetConfig, SPARK_PANDAS_CONVERSION_LIMIT=len(data)):
        local = distributed.to_backend(BackendsEnum.pandas)
        print(f"Round trip: {len(local)} rows on {local.backend_type.value}.")
        print("Outcome columns after conversion:", local.search_columns(TargetRole()))
finally:
    try:
        spark.stop()
    finally:
        spark_files.cleanup()

,segment,visits,count
0,new,0,3
1,new,2,3
2,returning,1,3
3,returning,3,3


to_backend conversion: 12 rows exceed limit 4
Round trip: 12 rows on pandas.
Outcome columns after conversion: ['revenue']
